### Data analysis
Reads all CSVs from `results/experiment_id`, concatenates them, saves under `processed_results/experiment_id`, computes mean features per well, and plots area in plate view.

### Imports

In [ ]:
import os
from pathlib import Path
import pandas as pd
from utils_data_analysis import plot_plate_view

### Config

In [ ]:
# Experiment ID (must match the folder name under results/)
experiment_id = "SK0047_Exp01-02"

results_dir = Path("results") / experiment_id
processed_dir = Path("processed_results") / experiment_id
os.makedirs(processed_dir, exist_ok=True)
print(f"Reading from {results_dir}")
print(f"Writing to {processed_dir}")

### Read all CSVs, concatenate, and save

In [ ]:
csv_files = sorted(results_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV files in {results_dir}")

df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
out_path = processed_dir / "concatenated.csv"
df_all.to_csv(out_path, index=False)
print(f"Concatenated {len(csv_files)} files → {len(df_all)} rows. Saved to {out_path}")

### Mean features per well and plate view of feature

In [ ]:
# Average all numeric features by well_id
df_mean = df_all.groupby("well_id", as_index=False).mean(numeric_only=True)

# Save mean dataframe
mean_path = processed_dir / "mean_per_well.csv"
df_mean.to_csv(mean_path, index=False)
print(f"Mean per well saved to {mean_path}")

In [ ]:
for column in df_mean.columns:
    print(column)

In [ ]:
for feature in df_mean.columns:

    if feature == 'well_id' or feature == 'FOV' or feature == 'label':
        pass
    else:    
        # Plot feature in plate view
        plot_plate_view(
            df_mean.copy(),
            column_name=feature,
            title=feature,
            label=feature,
            save_dir=str(processed_dir),
            fmt=0,
            display=False,
        )